# Functional test: does `@udf_init_once` work in a Python UDTF?
*Author: Garrett Frere (garrett.frere@snowflake.com)*

Two UDTFs with the same logic run on 20 stores (~2M rows) of `STORE_SALES` on `UDXF_S_WH` (2 nodes, 16 workers):
- **A** - expensive init (5s sleep) decorated with `@udf_init_once`
- **B** - same init as plain module-level code (control)

Each partition reports the process that handled it (`WORKER_PID`) and the process where the init ran (`INIT_PID`).
- init_once works -> few `INIT_PID`s (one parent per node), each shared by many workers
- per-worker load -> `INIT_PID == WORKER_PID`, one init per worker

In [ ]:
import os
import time
import pandas as pd
from snowflake.snowpark.context import get_active_session
from snowflake.snowpark.functions import col
from snowflake.snowpark.types import LongType, FloatType, StructType, StructField

session = get_active_session()
session.use_database("ML_PERF")
session.use_schema("PUBLIC")
session.use_warehouse("UDXF_S_WH")
session.sql("ALTER SESSION SET USE_CACHED_RESULT = FALSE").collect()

STAGE = "@ML_PERF.PUBLIC.UDXF_STAGE"
HANDLER_DIR = "udtf_init_once_handlers"
os.makedirs(HANDLER_DIR, exist_ok=True)

subset_df = session.table("ML_PERF.PUBLIC.STORE_SALES").filter(col("STORE_ID") <= 20)
print(f"rows: {subset_df.count():,}")

In [ ]:
# Same UDTF body for both; only the init differs
body = '''
class Handler:
    def __init__(self):
        self.store = None

    def process(self, store_id):
        self.store = store_id

    def end_partition(self):
        yield (self.store, os.getpid(), STATE["pid"], STATE["ts"])
'''

src_a = '''
import os, time
from _snowflake import udf_init_once

STATE = {}

@udf_init_once
def load():
    time.sleep(5)  # simulated expensive load
    STATE["pid"] = os.getpid()
    STATE["ts"] = time.time()
''' + body

src_b = '''
import os, time

STATE = {}
time.sleep(5)  # simulated expensive load, plain module-level code
STATE["pid"] = os.getpid()
STATE["ts"] = time.time()
''' + body

for name, src in [("udtf_a.py", src_a), ("udtf_b.py", src_b)]:
    with open(os.path.join(HANDLER_DIR, name), "w") as f:
        f.write(src)
print(os.listdir(HANDLER_DIR))

In [ ]:
output_schema = StructType([
    StructField("STORE", LongType()),
    StructField("WORKER_PID", LongType()),
    StructField("INIT_PID", LongType()),
    StructField("INIT_TS", FloatType()),
])

udtfs = {}
for label, fname, obj in [("A_init_once", "udtf_a.py", "UDTF_INIT_ONCE_TEST"), ("B_module_load", "udtf_b.py", "UDTF_MODULE_LOAD_TEST")]:
    try:
        udtfs[label] = session.udtf.register_from_file(
            file_path=os.path.join(HANDLER_DIR, fname),
            handler_name="Handler",
            name=f"ML_PERF.PUBLIC.{obj}",
            output_schema=output_schema,
            input_types=[LongType()],
            is_permanent=True,
            stage_location=STAGE,
            replace=True,
            packages=["snowflake-snowpark-python"],
        )
        print(f"{label}: registered")
    except Exception as e:
        print(f"{label}: REGISTRATION FAILED -> {e}")

In [ ]:
summary = []
for label, fn in udtfs.items():
    t0 = time.time()
    try:
        pdf = subset_df.select(fn(col("STORE_ID")).over(partition_by="STORE_ID")).to_pandas()
    except Exception as e:
        print(f"{label}: RUN FAILED -> {e}")
        continue
    summary.append({
        "udtf": label,
        "partitions": len(pdf),
        "worker_pids": pdf["WORKER_PID"].nunique(),
        "init_runs": pdf["INIT_PID"].nunique(),
        "init_pid_is_worker_pid": bool((pdf["INIT_PID"] == pdf["WORKER_PID"]).all()),
        "seconds": round(time.time() - t0, 1),
        "query_id": session.sql("SELECT LAST_QUERY_ID()").collect()[0][0],
    })
    print(summary[-1])

session.sql("ALTER SESSION UNSET USE_CACHED_RESULT").collect()
session.use_warehouse("ML_MODEL_WH")
pd.DataFrame(summary)

## Result: `@udf_init_once` works in a Python UDTF
| UDTF | partitions | worker PIDs | init runs | init in worker? |
|---|---|---|---|---|
| A - `@udf_init_once` | 20 | 10 | **1** | No (parent process) |
| B - module-level | 20 | 10 | **10** | Yes (every worker) |

- **A:** the 5s load ran **once**, in a parent process, and all 10 workers shared it.
- **B:** the same load ran **10 times**, once inside each worker, so there were 10 copies in memory.

Wall time is similar here (12.5s vs 11.3s) because B's 10 loads run in parallel. The win from init_once is **one shared copy in memory instead of one per worker**, which matters for large models.